# New Applicant Batch Inference

Loads raw tables for a batch of new applicants, builds features via the
shared inference package (`features/pipeline.py`, `model.py`,
`pdo_scorecard.py`, `score_model_input.py`), scores them against the
registered `HomeCredit_Scorecard_Model` (`@prod` alias), and converts
predictions to PDO credit scores and A-E risk bands.

This notebook is intentionally thin — all real logic (feature assembly,
model loading, PDO transform) lives in that shared package, the same one
`score_model_input.py` (Airflow) and `realtime_api.py` (FastAPI) both call.
That's deliberate: if this notebook re-implemented any of that logic
instead of importing it, it would drift from what batch/API scoring
actually do, the same way this notebook drifting from
`Credit_Scoring.ipynb`'s PDO formula would silently produce a second,
inconsistent scoring scale.

**Pipeline overview**
1. Configuration — paths, run date
2. Load raw tables for new applicants
3. Assemble features (shared feature contract)
4. Load model & predict
5. Convert PDs to scores + risk bands
6. Sanity-check the batch (band distribution)
7. Save outputs

**Prerequisite:** `HomeCredit_Scorecard_Model` must have a version with
the `prod` alias assigned (done manually in the MLflow UI after review —
see `02_training_LightGBM_v3.ipynb` section 8), and `selected_features.xlsx`
must exist.

**Outputs**
- `data/Inference/BatchOutput/<run_date>/scored_applicants.parquet`
- `data/Inference/BatchOutput/<run_date>/band_summary.csv`

## 1. Imports & Configuration

Same env-var pattern as your other notebooks (`HOMECREDIT_DATA_ROOT`,
`HOMECREDIT_SRC_DIR`) so this drops into the same project layout without
edits. `SRC_ROOT` here is expected to contain the inference package
(`src/Inference/`) on `sys.path`.

In [9]:
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = os.environ.get("HOMECREDIT_PROJECT_ROOT", r"a:\Dominic\My Projects\Home Credit")
DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")

# Path to the inference package's `src` directory.
INFERENCE_PKG_SRC = os.environ.get("HOME_CREDIT_INFERENCE_SRC", os.path.join(SRC_ROOT, "Inference"))
sys.path.insert(0, INFERENCE_PKG_SRC)

import config  # noqa: E402
from score_model_input import score_applicants  # noqa: E402

# ── Which batch of new applicants to score ──────────────────────────────────
RUN_DATE = "2026-07-31"  # change per run, or set from a notebook parameter

RAW_DATA_DIR = Path(DATA_ROOT) / "Input"
APP_DATA_DIR = Path(DATA_ROOT) / "Inference/BatchInput" / RUN_DATE
OUTPUT_DIR = Path(DATA_ROOT) / "Inference/BatchOutput" / RUN_DATE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reading raw applicant tables from : {RAW_DATA_DIR}")
print(f"Writing scored output to           : {OUTPUT_DIR}")
print(f"Model                              : {config.MODEL_NAME}@{config.MODEL_ALIAS}")
print(f"Feature list                       : {config.SELECTED_FEATURES_PATH}")

Reading raw applicant tables from : a:\Dominic\My Projects\Home Credit\data\Input
Writing scored output to           : a:\Dominic\My Projects\Home Credit\data\Inference\BatchOutput\2026-07-31
Model                              : HomeCredit_Scorecard_Model@prod
Feature list                       : a:\Dominic\My Projects\Home Credit\src\Inference\selected_features.xlsx


## 2. Load Raw Tables for New Applicants

Calls `score_model_input.load_raw_tables()` -- the same loader
`score_model_input.py` (CLI/Airflow) uses -- instead of reimplementing
the loading loop here, so this notebook can't drift from what batch
scoring actually reads.

Only `application`, `bureau`, `previous_application`, `pos_cash`, and
`installments` are loaded — `bureau_balance` and `credit_card` are skipped
entirely since neither is part of final model attributes
(`credit_card.py` is still a TODO stub), so reading them would just be
wasted I/O on two of the largest raw tables.

Each table is also read with `usecols=RAW_COLUMN_REQUIREMENTS[key]` —
the exact raw columns the corresponding `features/*.py` module consumes
(see `features/pipeline.py::RAW_COLUMN_REQUIREMENTS`) — instead of every
raw column. `bureau.csv`/`previous_application.csv`/`installments_payments.csv`
are multi-million-row files where most columns are never touched by
feature engineering, so this is a large, free memory/IO win.

`application.csv` is required (`load_raw_tables` raises `FileNotFoundError`
if it's missing); the rest are optional — if a table isn't present, its
aggregates are simply skipped in feature assembly (see `assemble_features`
docstring). Missing a table your model actually relies on will surface as
a clear error in the next cell, not a silent bad prediction.


In [10]:
from score_model_input import load_raw_tables

raw_tables = load_raw_tables(APP_DATA_DIR, RAW_DATA_DIR)
print(f"\nScoring {len(raw_tables['application']):,} new applicants")


2026-08-05 23:36:03,566 INFO score_model_input: Loaded application            (48744, 14)
2026-08-05 23:36:05,082 INFO score_model_input: Loaded bureau                 (1716428, 9)
2026-08-05 23:36:07,920 INFO score_model_input: Loaded previous_application   (1670214, 6)
2026-08-05 23:36:11,111 INFO score_model_input: Loaded pos_cash               (10001358, 4)
2026-08-05 23:36:18,091 INFO score_model_input: Loaded installments           (13605401, 6)



Scoring 48,744 new applicants


## 3. Assemble Features

Calls the same `assemble_features()` used by batch scoring and the API.
Raises immediately, with the specific missing column names, if a raw
table wasn't supplied or a feature module is still a placeholder — this
is the point where a silently-wrong feature matrix would otherwise reach
the model.


In [11]:
from features.pipeline import assemble_features

feature_df = assemble_features(raw_tables)
print(f"Feature matrix shape: {feature_df.shape}")
feature_df.head(3)

Columns containing sentinel 365243: ['DAYS_EMPLOYED']
Columns containing 'XNA': ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']
Credit-sum imputation complete.
Remaining nulls in imputed columns:
AMT_CREDIT_SUM_LIMIT     726
AMT_CREDIT_SUM_DEBT     3178
Derived UTILIZATION_RATIO.
Aggregating features by SK_ID_CURR...
Aggregating count-based features...
Final aggregated features:
42320 applicants with aggregated bureau features.
Columns containing sentinel 365243: []
Sentinel values replaced with NaN.
Columns containing 'XNA': []
XNA values replaced with NaN.
Feature matrix shape: (48744, 26)


,SK_ID_CURR,AGE,BUREAUCREDIT_AMT_CREDIT_SUM_DEBT_MEAN,BUREAUCREDIT_AVG_DAYS_BETWEEN_CREDITS,BUREAUCREDIT_DAYS_CREDIT_UPDATE_MEAN,BUREAUCREDIT_DAYS_TO_LAST_CREDIT,BUREAUCREDIT_TOTAL_ACTIVE_CREDIT_COUNT,BUREAUCREDIT_UTILIZATION_RATIO_MEAN,DAYS_ID_PUBLISH,EMPLOYED_YEARS,...,ORGANIZATION_TYPE,OWN_CAR_AGE,PMNT_INSTLMT_AVG_CREDIT_TO_APPL,PMNT_INSTLMT_LATE_PAYMENT_DAYS_MEAN_1Y,PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO_MEAN_2Y,POS_MONTHS_SINCE_LAST_DPD_GT0,PREVIOUS_REFUSED_LOAN_COUNT,PREVIOUS_REFUSED_LOAN_LAST_REFUSED_DAY,PREVIOUS_TOTAL_LOAN_CREDIT_TO_APPL_MEAN_LAST3Y,YEARS_SINCE_PHONE_CHANGE
0,100001,52.715069,85240.929688,253.833328,-93.142860,-49.0,3.0,0.282518,-812,6.380822,...,Kindergarten,NaN,0.957782,NaN,NaN,95.0,NaN,NaN,NaN,4.767123
1,100005,49.490410,189469.500000,155.500000,-54.333332,-62.0,2.0,0.601256,-1623,12.243835,...,Self-employed,NaN,0.899950,NaN,1.0,NaN,NaN,NaN,0.899950,-0.000000
2,100013,54.898628,0.000000,286.666656,-775.500000,-1210.0,0.0,0.000000,-3503,12.213698,...,Transport: type 3,5.0,1.052363,3.333333,1.0,4.0,NaN,NaN,1.108879,2.345206


## 4. Load Model & Predict

Loads `HomeCredit_Scorecard_Model@prod` from the MLflow registry (cached
after first call, so re-running this cell in the same kernel session
doesn't reload it) and scores the batch.


In [12]:
scored_df = score_applicants(feature_df)
print(f"Scored {len(scored_df):,} applicants")
print(f"PD range    : [{scored_df['PROBA_DEFAULT'].min():.4f}, {scored_df['PROBA_DEFAULT'].max():.4f}]")
print(f"Score range : [{scored_df['score'].min()}, {scored_df['score'].max()}]")
scored_df.head(3)

Scored 48,744 applicants
PD range    : [0.0022, 0.7917]
Score range : [449, 664]


,SK_ID_CURR,PROBA_DEFAULT,score,Risk_Band
0,100001,0.049694,572,D - High Risk
1,100005,0.165238,534,E - Very High Risk
2,100013,0.062054,565,D - High Risk


## 5. Sanity-Check the Batch — Risk Band Distribution

Before saving, eyeball the band split for this batch. A batch that's
wildly different from your validation-set band distribution (see
`Credit_Scoring.ipynb` section 5) is worth a second look before it feeds
downstream decisions — could be a genuinely different applicant pool, or
could be a feature-assembly bug upstream.


In [13]:
band_summary = (
    scored_df.groupby("Risk_Band", observed=True)
    .agg(Applicant_Count=("SK_ID_CURR", "count"), Mean_Score=("score", "mean"), Mean_PD=("PROBA_DEFAULT", "mean"))
    .reset_index()
)
band_summary["Volume_Pct"] = band_summary["Applicant_Count"] / band_summary["Applicant_Count"].sum()

print(band_summary.to_string(index=False, float_format="{:.4f}".format))

         Risk_Band  Applicant_Count  Mean_Score  Mean_PD  Volume_Pct
E - Very High Risk            16946    539.4396   0.1532      0.3477
     D - High Risk            19022    574.3906   0.0481      0.3902
   C - Medium Risk            11231    601.5500   0.0193      0.2304
      B - Low Risk             1524    627.3878   0.0079      0.0313
 A - Very Low Risk               21    653.8571   0.0031      0.0004


## 6. Save Outputs


In [7]:
scored_path = OUTPUT_DIR / "scored_applicants.parquet"
scored_df.to_parquet(scored_path, index=False)

band_summary_path = OUTPUT_DIR / "band_summary.csv"
band_summary.to_csv(band_summary_path, index=False)

print(f"Saved scored applicants to : {scored_path}")
print(f"Saved band summary to      : {band_summary_path}")

Saved scored applicants to : a:\Dominic\My Projects\Home Credit\data\Inference\BatchOutput\2026-07-31\scored_applicants.parquet
Saved band summary to      : a:\Dominic\My Projects\Home Credit\data\Inference\BatchOutput\2026-07-31\band_summary.csv


## 7. Cleanup


In [8]:
del raw_tables, feature_df, scored_df
print("Done.")

Done.
